# Tunisia travel recommendation (24 governorates)

This notebook regenerates the **logical synthetic dataset**, trains **HistGradientBoostingClassifier + one-hot pipeline**, exports:
- `frontend/src/assets/tunisian_travel_dataset_24c.csv`
- `frontend/src/assets/travel-match-model.json` (browser fallback via centroids)
- `travel-recommendation/model_bundle.joblib` (Flask API)

**Accuracy:** Labels are anchored by `(preferred_region, travel_style)` with ~1.5% noise, so hold-out accuracy should be ~97–99%.

**Prerequisite:** Python 3.11+ with `pandas`, `numpy`, `scikit-learn`, `joblib` (`pip install -r travel-recommendation/requirements.txt` installs Flask too).

In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "scripts" / "generate_tunisia_travel_dataset.py").is_file():
    ROOT = ROOT.parent
assert (ROOT / "scripts" / "generate_tunisia_travel_dataset.py").is_file(), "Run from repo root or notebooks/"

subprocess.check_call([sys.executable, str(ROOT / "scripts" / "generate_tunisia_travel_dataset.py"), "--rows", "4800", "--noise", "0.015"])
subprocess.check_call([sys.executable, str(ROOT / "scripts" / "train_travel_match_model.py")])
print("Done — model JSON + joblib written.")

## Serve recommendations with Flask

From repo root (separate terminal):

```bash
pip install -r travel-recommendation/requirements.txt
python travel-recommendation/app.py
```

Angular dev (`ng serve`): requests go to `/travel-rec/api/recommend` → proxies to `http://127.0.0.1:5050` (see `frontend/src/proxy.conf.json`). Set `travelRecommendationApiUrl` to `""` in `environment.development.ts` to force static JSON-only mode.

**Health:** `GET http://127.0.0.1:5050/health`

In [ ]:
# Optional smoke test when Flask is running locally
import urllib.request
import json

payload = {
    "age": 34,
    "gender": "female",
    "nationality": "tunisian",
    "current_city": "Tunis",
    "travel_style": "cultural",
    "travel_styles": ["cultural"],
    "budget_level": "medium",
    "preferred_region": "north",
    "preferred_cuisine": "tunisian",
    "travel_with": "couple",
    "transport_preference": "car",
    "accommodation_type": "hotel",
    "travel_intensity": "medium",
    "budget_avg": 180,
    "is_group": 0,
    "topN": 5,
}

try:
    req = urllib.request.Request(
        "http://127.0.0.1:5050/api/recommend",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=3) as resp:
        print(json.dumps(json.loads(resp.read().decode()), indent=2))
except Exception as e:
    print("Flask not running or unreachable:", e)